# Práctica 1 · Un sistema RAG mínimo, de principio a fin

Un modelo de lenguaje no conoce tus documentos. Sabe mucho de lo que había en internet cuando
lo entrenaron, pero no sabe nada del manual de tu empresa, ni de tus políticas de devolución,
ni de las preguntas frecuentes de tus clientes. Si le preguntas por eso, o admite que no sabe,
o se lo inventa.

RAG resuelve ese problema sin reentrenar nada. La idea es simple: antes de responder, buscamos
en nuestros propios documentos los fragmentos relacionados con la pregunta, se los entregamos
al modelo, y le pedimos que responda usando solo eso.

Hoy vas a construir ese circuito completo. Vamos a tomar un libro en PDF, partirlo en
fragmentos, guardarlos de manera que se puedan buscar por significado y no por palabras
exactas, y armar la cadena que responde preguntas sobre él.

Todo corre dentro de tu computadora. No hay cuenta que abrir, ni clave de acceso, ni cobro por
consulta: los modelos se ejecutan en tu propia máquina a través de Ollama.

Necesitas Ollama instalado y los modelos descargados. Si aún no lo has hecho, abre
`INSTALACION.md` y sigue los pasos antes de continuar.

Las celdas se ejecutan en orden, una por una, con Shift + Enter. Cada una usa lo que definió
la anterior, así que no te saltes ninguna.

### Una aclaración, antes de empezar

Lo que vas a construir esta semana no es, ni pretende ser, un chatbot comercial de alto
rendimiento. Esos corren sobre modelos cientos de veces más grandes que el de tu laptop, con
equipos que mantienen el contenido al día, integraciones con los sistemas de la empresa y una
operación que no se apaga. La diferencia de desempeño es real y no se cierra en cinco días.

Lo que sí es igual en los dos: la arquitectura, que es la misma pieza por pieza; los modos en
que fallan, que no se resuelven haciendo el modelo más grande; y la forma de medirlos, que es
la parte que de verdad te vas a llevar. Al terminar no vas a tener un producto que compita con
un servicio de pago, pero vas a poder abrir uno, preguntarle lo correcto y saber si sirve.

Este es el circuito completo que vas a tener funcionando al final de la sesión. Vale la pena
mirarlo un momento ahora, y volver a él cuando te pierdas entre las celdas.

<img src="figuras/01_arquitectura_rag.svg" alt="Diagrama del flujo de un sistema RAG: la pregunta busca en los documentos indexados, los fragmentos más parecidos y la pregunta llegan al modelo, y el modelo responde usando solo ese contexto." width="740">

Fíjate en la línea punteada. La pregunta hace dos viajes: uno para ir a buscar en los
documentos, y otro para llegar al modelo junto con lo que se encontró. Esa segunda vez el
modelo ya tiene delante los fragmentos del libro, y por eso puede responder sobre algo que
nunca memorizó.

## 1. Instalar las librerías de Python

Cinco librerías hacen el trabajo. `langchain-community` y `langchain-ollama` son el pegamento
que conecta las piezas. `langchain-text-splitters` parte documentos largos en fragmentos.
`lancedb` es la base de datos donde se guardan esos fragmentos para poder buscarlos. `pypdf`
lee archivos PDF.

Si ya las instalaste al preparar tu equipo, la celda termina de inmediato sin hacer nada.
Ejecútala de todas formas, para estar seguro de que no falta ninguna.

In [1]:
%pip install --quiet langchain-community langchain-ollama langchain-text-splitters lancedb pypdf

print("Librerías listas.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Librerías listas.


## 2. Comprobar que Ollama responde

Ollama es el programa que ejecuta los modelos de lenguaje dentro de tu computadora. Tiene que
estar encendido para que este cuaderno funcione, así que lo primero es confirmarlo.

Si algo falla, la salida de la celda te dice qué hacer según tu sistema operativo.

In [2]:
import platform
import sys

import requests

OLLAMA_URL = "http://localhost:11434"

print(f"Sistema: {platform.system()} {platform.machine()}")
print(f"Python:  {sys.version.split()[0]}\n")

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    respuesta.raise_for_status()
    modelos = sorted(m["name"] for m in respuesta.json()["models"])
    print(f"Ollama responde. Tienes {len(modelos)} modelos descargados:\n")
    for m in modelos:
        print(f"  - {m}")
except Exception as e:
    print(f"Ollama no responde en {OLLAMA_URL}")
    print(f"Detalle: {type(e).__name__}\n")
    if platform.system() == "Darwin":
        print("En Mac: abre la aplicación Ollama desde la carpeta Aplicaciones.")
        print("Debe aparecer su ícono en la barra de menús, arriba a la derecha.")
    elif platform.system() == "Windows":
        print("En Windows: busca Ollama en el menú Inicio y ábrelo.")
        print("Debe aparecer su ícono junto al reloj, abajo a la derecha.")
    else:
        print("Ejecuta 'ollama serve' en una terminal.")

Sistema: Darwin arm64
Python:  3.12.13



Ollama responde. Tienes 14 modelos descargados:

  - embeddinggemma:300m
  - gemma3:1b
  - gemma3:4b
  - gemma4:12b-mlx
  - gemma4:26b
  - gemma4:e4b
  - granite4.1:3b
  - granite4.1:8b
  - mxbai-embed-large:latest
  - nemotron-mini:4b
  - nomic-embed-text:latest
  - qwen3-4b-cs-ft:latest
  - qwen3:4b
  - shieldgemma:2b


## 3. Elegir el modelo según tu equipo

El modelo va a correr en tu máquina, así que la memoria que tengas importa. Un modelo grande
en un equipo chico no se rompe: simplemente tarda muchísimo y el sistema se pone lento.

Abajo hay tres opciones. Deja activa una sola, la que corresponda a tu computadora, y comenta
las demás poniéndoles un signo de gato al inicio de la línea. Si no sabes cuánta memoria
tienes, quédate con la opción A, que funciona en cualquier equipo.

El modelo de embeddings no se elige por equipo: es ligero y va igual en todos. Sí conviene saber
de dónde salió esa elección, y la respuesta es que está medida con documentos en español; en la
práctica 3 vas a reproducir la medición y a ver a los tres candidatos compitiendo.

In [3]:
# ---- Opción A: equipos de 8 GB de memoria o menos (descarga 3.3 GB) ---------
MODELO_LLM = "gemma3:4b"

# ---- Opción B: equipos de 16 GB de memoria (descarga 10 GB) ----------------
# MODELO_LLM = "gemma4:12b"        # Windows y Linux
# MODELO_LLM = "gemma4:12b-mlx"    # Mac con chip Apple (M1 en adelante), va más rápido

# ---- Opción C: equipos de 32 GB de memoria o más (descarga 17 GB) ----------
# MODELO_LLM = "gemma4:26b"        # Windows y Linux
# MODELO_LLM = "gemma4:26b-mlx"    # Mac con chip Apple

# El modelo de embeddings es ligero y es el mismo para todos. La elección está
# medida, no copiada de un tutorial: lo comprobamos en la práctica 3. Se eligió
# éste porque es el único de los tres que encuentra un pasaje en inglés cuando la
# pregunta va en español, algo que hace falta en cuanto el corpus mezcla idiomas.
MODELO_EMBEDDINGS = "embeddinggemma:300m"

print(f"Modelo de lenguaje:   {MODELO_LLM}")
print(f"Modelo de embeddings: {MODELO_EMBEDDINGS}")
print("\nSi alguno no aparece en la lista de la celda anterior, descárgalo con:")
print(f"   ollama pull {MODELO_LLM}")
print(f"   ollama pull {MODELO_EMBEDDINGS}")

Modelo de lenguaje:   gemma3:4b
Modelo de embeddings: embeddinggemma:300m

Si alguno no aparece en la lista de la celda anterior, descárgalo con:
   ollama pull gemma3:4b
   ollama pull embeddinggemma:300m


## 4. Importar lo que vamos a usar

Conviene saber qué hace cada pieza antes de ejecutarlas.

`PyPDFLoader` abre el PDF y saca el texto. `RecursiveCharacterTextSplitter` lo parte en
fragmentos. `OllamaEmbeddings` convierte texto en números y `ChatOllama` es el modelo que
redacta las respuestas. `LanceDB` guarda los fragmentos y sabe buscar entre ellos.
`ChatPromptTemplate` arma las instrucciones que recibirá el modelo, `StrOutputParser` se queda
con el texto de la respuesta y `RunnablePassthrough` deja pasar la pregunta sin modificarla.

Las dos primeras líneas apagan un aviso en rojo que sale al importar y que no es un error:
avisa que esa librería se está repartiendo en paquetes más chicos. Todo funciona igual.

In [4]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import LanceDB
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Listo.")

Listo.


## 5. Cargar el documento

Vamos a trabajar con *Alicia en el país de las maravillas*, de Lewis Carroll. Es un texto
conocido, está en inglés y es lo bastante largo como para que la búsqueda tenga que trabajar de
verdad.

El PDF viene en la carpeta `documentos`, junto a este cuaderno. Lo cargamos de ahí y no de
internet, por una razón práctica: el servidor que lo hospeda corta las descargas cuando recibe
varias seguidas desde la misma red, y en un salón entero pidiéndolo a la vez es casi seguro que
falle. Si algún día lo necesitas, la celda lo baja sola cuando el archivo no está.

`PyPDFLoader` convierte el archivo en una lista de objetos `Document`: uno por página, cada uno
con su texto y sus datos de origen.

In [5]:
# Path sirve para nombrar archivos y carpetas sin preocuparse de si el sistema
# usa barras o contrabarras. Funciona igual en Mac y en Windows.
from pathlib import Path

PDF = Path("documentos/Alice_in_Wonderland.pdf")

# Si el archivo ya está en la carpeta, esta parte se salta sola.
if not PDF.exists():
    print("No encontré el PDF en la carpeta. Descargándolo...")
    PDF.parent.mkdir(exist_ok=True)
    r = requests.get(
        "https://www.adobe.com/be_en/active-use/pdf/Alice_in_Wonderland.pdf",
        timeout=180,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    r.raise_for_status()
    PDF.write_bytes(r.content)
    print(f"Guardado en {PDF}")

# PyPDFLoader es el lector de PDF. Se le dice qué archivo abrir...
loader = PyPDFLoader(str(PDF))
# ...y load() devuelve una lista con una entrada por página del documento.
pages = loader.load()

print(f"Páginas cargadas: {len(pages)}")

Páginas cargadas: 105


Antes de seguir, asómate a lo que quedó cargado. Es un hábito que vale oro: casi todos los
problemas de un sistema RAG se ven aquí, en el texto de entrada, mucho antes de llegar al
modelo.

Fíjate en dos cosas. El texto trae los cortes de línea del PDF, no los del libro. Y arrastra
elementos que no son parte de la novela, como el pie de página del programa con el que se armó
ese archivo. Guarda esa observación, porque más adelante en el curso vamos a medir cuánto
afecta a la calidad de las respuestas.

In [6]:
print(pages[0].metadata)
print()
print(pages[10].page_content[:500])

{'producer': 'BookVirtual Corp. Patents Pending.', 'creator': 'BookVirtual Digital Works', 'creationdate': '2000', 'keywords': "Carroll, Alice, Wonderland, children's, 1865; v.1.2", 'title': "Alice's Adventures in Wonderland", 'moddate': '2000-11-27T16:31:36-08:00', 'subject': "children's literature", 'author': 'Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<n/a>;', 'source': 'documentos/Alice_in_Wonderland.pdf', 'total_pages': 105, 'page': 0, 'page_label': '1'}

Alice was not a bit hurt, and she jumped up
on to her feet in a moment : she looked up,
but it was all dark overhead ; before her was
another long passage, and the White Rabbit was
still in sight, hurrying down it. There was
not a moment to be lost : away went Alice like
the wind, and was just in time to hear it say, as
it turned a corner, “ Oh my ears and whiskers,
how late it ’s getting !” She was close behind
it when she turned the corner, but the Rabbit
was no longer to be seen : she found h


## 6. Quitar la basura del PDF

Ya viste que cada página arrastra el pie de página del visor y las etiquetas de sus botones.
Eso no es parte de la novela y conviene quitarlo antes de seguir, por dos razones concretas.

La primera es que ocupa lugar. Cada fragmento tiene un presupuesto de 1000 caracteres, y ese
pie mide 87. Como se repite en las 105 páginas, acaba metido en casi todos los fragmentos: en
un contexto de tres fragmentos son cerca de 260 caracteres desperdiciados, casi un 9% del texto
que le entregamos al modelo, gastado en un aviso de copyright.

La segunda es que ensucia la búsqueda. Ese texto también se convierte en vectores, y algo que
aparece idéntico en todas las páginas no ayuda a distinguir una de otra.

Los patrones de abajo están hechos a la medida de este PDF. Con otro documento tendrás que
mirar qué basura repetida trae y ajustar la lista.

In [7]:
import re

RUIDO = [
    r"Digital Interface by BookVirtual Corp\. U\.S\. Patent Pending\.\s*[©']?\s*2000 All Rights Reserved\.",
    r"Fit Page\s+Full Screen\s+On/Off\s+Close Book",
    r"Navigate\s+Control\s+Internet",
    r"CLOSE THE BOOK|TURN THE PAGE|NAVIGATE|CONTROL",
]


def limpiar(texto):
    for patron in RUIDO:
        texto = re.sub(patron, " ", texto)
    return re.sub(r"[ \t]{2,}", " ", texto).strip()


paginas_limpias = []
for pagina in pages:
    copia = pagina.model_copy(deep=True)
    copia.page_content = limpiar(pagina.page_content)
    paginas_limpias.append(copia)

antes = sum(len(p.page_content) for p in pages)
despues = sum(len(p.page_content) for p in paginas_limpias)
print(f"Caracteres antes:   {antes}")
print(f"Caracteres después: {despues}")
print(f"Eliminado: {antes - despues} caracteres ({100 * (antes - despues) / antes:.1f}%)")
print("\nLa página 10 ahora empieza así:\n")
print(paginas_limpias[10].page_content[:300])

Caracteres antes:   170686
Caracteres después: 153749
Eliminado: 16937 caracteres (9.9%)

La página 10 ahora empieza así:

Alice was not a bit hurt, and she jumped up
on to her feet in a moment : she looked up,
but it was all dark overhead ; before her was
another long passage, and the White Rabbit was
still in sight, hurrying down it. There was
not a moment to be lost : away went Alice like
the wind, and was just in ti


Una advertencia que vale más que la limpieza misma.

Lo primero que uno quiere hacer también es unir los saltos de línea del PDF, para que el texto
se lea corrido en lugar de cortado. Lo probé, y hay que resistir la tentación: al unirlos, el
número de fragmentos cambia y con él las fronteras donde se parte el texto. El resultado es que
la búsqueda empieza a fallar en preguntas que antes acertaba. En una de las pruebas que vamos a
hacer en la sección 12, el sistema dejaba de encontrar el pasaje correcto.

Por eso aquí solo quitamos la basura y dejamos los saltos de línea en paz.

Guárdate la lección, porque es de las que cuestan caro: en un sistema RAG, cambiar el
preprocesamiento cambia dónde se corta cada fragmento, y eso reordena los resultados de la
búsqueda de formas que no se pueden anticipar leyendo el código. Cualquier ajuste de este paso
hay que medirlo, no razonarlo.

## 7. Partir el texto en fragmentos

No podemos darle el libro completo al modelo en cada pregunta: no cabe, y aunque cupiera,
pagaríamos en tiempo lo que ganáramos en contexto. Además, mientras más texto irrelevante
reciba, más fácil es que se distraiga.

Por eso el documento se corta en fragmentos. `chunk_size=1000` fija el tamaño en caracteres.
`chunk_overlap=200` hace que cada fragmento repita el final del anterior; ese traslape evita
que una idea quede partida justo en la frontera entre dos pedazos y no aparezca completa en
ninguno de los dos.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # tamaño máximo de cada trozo, en caracteres
    chunk_overlap=200,    # cuánto repite del trozo anterior, para no cortar ideas
    length_function=len,  # cómo se mide el tamaño: aquí, contando caracteres
)
# split_documents recorre las páginas y devuelve la lista de trozos.
chunks = text_splitter.split_documents(paginas_limpias)

print(f"Fragmentos generados: {len(chunks)}")
print()
print(chunks[0].page_content)

Fragmentos generados: 221

BY LEWIS CARROLL ILLUSTRATED BY JOHN TENNIEL
 
A LICE ’S
Adventures in W onderland
 
 
 
The world’ s
most precise
replica
of the world’ s 
most famous
children’ s book!
In 1998, Peter Zelchenko
began a project for Volume-
One Publishing: to create an
exact digital replica of Lewis
Carroll’s ﬁrst edition of Alice.
Working with the original
1865 edition and numerous
other editions at the Newberry
Library in Chicago, Zelchenko
created a digital masterpiece in
his own right, a testament to
the original work of Lewis
Carroll (aka Prof. Charles
Dodgson) who personally
directed the typography for the
ﬁrst Alice.
After much analyis, Peter then
painstakingly matched letter to
letter, line to line, of his new
digital edition to that of the
original. After weeks of toil he
created an exact replica of the
original! The book was added
to VolumeOne’s print-on-
demand offering. While a PDF
version is offered on various
portals of the Net, BookVirtual
took the project to he

## 8. Convertir los fragmentos en vectores y guardarlos

Aquí está el corazón del asunto. El modelo de embeddings traduce cada fragmento a una lista
larga de números que representa su significado. Lo valioso es que dos textos que hablan de lo
mismo quedan cerca uno del otro aunque no compartan ni una palabra.

LanceDB guarda esos vectores y sabe encontrar los más cercanos a una consulta. Eso es lo que
permite buscar por significado en lugar de buscar por coincidencia exacta de palabras, que es
la diferencia entre un buscador que entiende la pregunta y uno que solo cuenta palabras.

Esta celda tarda: tu computadora tiene que procesar los fragmentos uno por uno. En un equipo
reciente son segundos; en uno más modesto, varios minutos. Es una sola vez, porque el
resultado queda guardado en la carpeta `lancedb`, junto a este cuaderno.

In [9]:
# El modelo de embeddings es el que traduce texto a números.
embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)

# from_documents hace tres cosas de una vez: convierte cada fragmento en su
# vector, crea la base de datos y guarda todo en la carpeta "lancedb", que
# aparecerá junto a este cuaderno.
vectorstore = LanceDB.from_documents(chunks, embeddings, uri="lancedb")

print("Índice creado.")

Índice creado.


## 9. Armar la cadena

Ya tenemos los fragmentos buscables. Falta conectar las piezas.

El recuperador es quien busca: le pedimos los tres fragmentos más parecidos a cada pregunta.
`temperature=0` hace que el modelo responda de forma estable, sin cambiar de una corrida a
otra. La plantilla del prompt es la instrucción que recibirá, y dice algo muy concreto:
responde basándote únicamente en este contexto.

El operador `|` encadena los pasos, como una tubería. Se lee de arriba abajo: recuperar el
contexto y darle formato, meterlo junto con la pregunta en la plantilla, mandarlo al modelo,
y quedarse solo con el texto de la respuesta.

In [10]:
# El modelo que redacta la respuesta. temperature=0 significa que ante la
# misma pregunta responderá siempre lo mismo, que es lo que se quiere en
# atención al cliente.
#
# Aquí es donde una organización pondría la llamada a un servicio comercial
# si decidiera no usar un modelo local. Este curso usa Ollama a propósito:
# sin costo por consulta y sin que los documentos salgan de la máquina.
llm = ChatOllama(model=MODELO_LLM, temperature=0)

# El buscador. k=3 significa que traerá los tres fragmentos más parecidos
# a cada pregunta. Ese número se ajusta midiendo, y lo veremos más adelante.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


def format_docs(docs):
    '''Pega los fragmentos recuperados en un solo texto, separados por
    una línea en blanco, que es como se los pasaremos al modelo.'''
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

# Esta es la cadena completa, y se lee de arriba hacia abajo. La barra
# vertical significa "y después"; funciona como una línea de montaje:
#
#   1. la pregunta entra y va por dos caminos a la vez:
#        - al buscador, que devuelve los fragmentos y los pega (context)
#        - tal cual, sin tocarla (question)
#   2. las dos piezas se meten en la plantilla del prompt
#   3. el prompt completo se le entrega al modelo
#   4. de la respuesta del modelo se extrae solo el texto
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Cadena lista.")

Cadena lista.


## 10. Preguntar

El momento de la verdad. La pregunta va en inglés porque el libro está en inglés.

La primera consulta tarda bastante más que las siguientes, porque Ollama tiene que cargar el
modelo en memoria. De la segunda en adelante ya está listo y responde más rápido.

In [11]:
query = "Describe the Mad Hatter's tea party."
answer = rag_chain.invoke(query)
print(answer)

Based on the text, here’s a description of the Mad Hatter’s tea party:

It’s chaotic and bizarre! Alice finds herself in a long hall with a little glass table where the March Hare and Hatter are hosting a tea party. There’s no wine, they make rude remarks (like commenting on Alice's hair), and the Hatter asks nonsensical questions (“Why is a raven like a writing-desk?”). The Dormouse falls asleep, and the Hatter tries to put the Dormouse in a teapot. At one point, he talks about manipulating time with the clock. It’s described as “the stupidest tea-party” Alice had ever been at.


Detente a pensar en lo que acaba de pasar. El modelo que corre en tu computadora no memorizó
*Alicia en el país de las maravillas*, y aun así describió con detalle una escena concreta del
libro. No lo hizo de memoria: lo hizo leyendo los tres fragmentos que le pasamos.

Ahí está la idea completa de RAG, y es lo que la vuelve tan útil en el trabajo real. El mismo
circuito, con los documentos de una empresa en lugar de una novela, responde preguntas sobre
políticas, manuales o procedimientos sin haber sido entrenado con nada de eso.

Compara tu respuesta con la de algún compañero que haya elegido un modelo distinto en la
sección 3. Los hechos deberían coincidir, porque ambos leyeron los mismos fragmentos, pero la
redacción va a ser diferente. Un modelo chico resume; uno grande desarrolla.

## 11. Mira lo que el modelo recibió

Vale la pena abrir la caja y ver exactamente qué se le entregó. Este es el mejor hábito que
puedes llevarte de hoy: cuando una respuesta salga mal, lo primero que hay que revisar no es el
modelo que redacta, sino los fragmentos que le llegaron.

Muchas veces la respuesta es mala simplemente porque la búsqueda trajo lo que no era, y ahí
cambiar de modelo no arregla nada.

Ojo con un detalle de esta celda: imprime solo los primeros 300 caracteres de cada fragmento,
para que la salida quepa en pantalla. El modelo recibió el fragmento completo, y el número de
la izquierda te dice de qué tamaño era en realidad.

In [12]:
docs = retriever.invoke(query)

print(f"El modelo recibió {len(docs)} fragmentos:\n")
for i, doc in enumerate(docs, 1):
    contenido = doc.page_content
    print(f"--- Fragmento {i} | página {doc.metadata.get('page', '?')} | "
          f"{len(contenido)} caracteres (se muestran 300) ---")
    print(contenido[:300].replace("\n", " "))
    print()

print(f"Contexto total entregado al modelo: {sum(len(d.page_content) for d in docs)} caracteres")

El modelo recibió 3 fragmentos:

--- Fragmento 1 | página 62 | 998 caracteres (se muestran 300) ---
A MAD TEA -PARTY.110  A MAD TEA -PARTY. 111 “Really, now you ask me,” said Alice, very much confused, “I don’t think——” “Then you shouldn’t talk,” said the Hatter. This piece of rudeness was more than Alice could bear : she got up in great disgust, and walked off : the Dormouse fell asleep instantly

--- Fragmento 2 | página 55 | 964 caracteres (se muestran 300) ---
A MAD TEA -PARTY.96  A MAD TEA -PARTY. 97 room !” said Alice indignantly, and she sat down in a large arm-chair at one end of the table. “Have some wine,” the March Hare said in an encouraging tone. Alice looked all round the table, but there was nothing on it but tea. “ I don ’t see any wine,” she 

--- Fragmento 3 | página 58 | 971 caracteres (se muestran 300) ---
A MAD TEA -PARTY.102  A MAD TEA -PARTY. 103 anything you liked with the clock. For in- stance, suppose it were nine o’clock in the morn- ing, just time to begin l

Y si quieres ver de verdad todo lo que recibió, sin recortes, aquí está el texto exacto que le
llegó: los fragmentos completos ya metidos en la plantilla del prompt, con la pregunta al final.

Esto es literalmente lo único que el modelo tenía enfrente cuando respondió. Nada más.

In [13]:
prompt_final = prompt.format(context=format_docs(docs), question=query)

print(f"El prompt completo mide {len(prompt_final)} caracteres.\n")
print("=" * 78)
print(prompt_final)
print("=" * 78)

El prompt completo mide 3050 caracteres.

Human: Answer the question based only on the following context:

A MAD TEA -PARTY.110
 A MAD TEA -PARTY. 111
“Really, now you ask me,” said Alice, very
much confused, “I don’t think——”
“Then you shouldn’t talk,” said the Hatter.
This piece of rudeness was more than Alice
could bear : she got up in great disgust, and
walked off : the Dormouse fell asleep instantly,
and neither of the others took the least notice
of her going, though she looked back once or
twice, half hoping that they would call after
her: the last time she saw them, they were
trying to put the Dormouse into the teapot.
At any rate I ’ll never go there again !” said
Alice as she picked her way through the wood.
“It ’s the stupidest tea-party I ever was at in
all my life !”
Just as she said this, she noticed that one
of the trees had a door leading right into it.
“ That ’s very curious !” she thought. “ But
everything’s curious to-day. I think I may as
well go in at once.” And in

## 12. La prueba que importa: el mismo modelo sin RAG

Hasta aquí construiste el sistema, pero todavía no has comprobado que sirva de algo. A lo mejor
el modelo ya se sabía el libro y los fragmentos no aportaron nada.

Vamos a averiguarlo. La función de abajo hace la misma pregunta dos veces al mismo modelo, en la
misma computadora. La única diferencia es que la primera vez responde solo, de memoria, y la
segunda recibe los fragmentos que encontró la búsqueda.

Esta sección usa `gemma3:4b` aunque hayas elegido otro modelo arriba, y lo hace a propósito. Es
el modelo que todos tenemos, así que la comparación sale igual en todas las pantallas del salón
y podemos discutirla juntos. Al final de la sección te cuento qué cambia con un modelo mayor.

In [14]:
MODELO_COMPARACION = "gemma3:4b"

llm_chico = ChatOllama(model=MODELO_COMPARACION, temperature=0)

# La misma cadena de la sección 9, pero con el modelo de la comparación.
# Así lo único que cambia entre las dos respuestas son los fragmentos.
cadena_chica = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_chico
    | StrOutputParser()
)


def comparar(pregunta):
    """Hace la misma pregunta dos veces: sin contexto y con contexto."""
    sin_rag = llm_chico.invoke(pregunta).content.strip()
    fuentes = retriever.invoke(pregunta)
    con_rag = cadena_chica.invoke(pregunta).strip()

    print(f"PREGUNTA: {pregunta}")
    print(f"MODELO:   {MODELO_COMPARACION} en los dos casos")
    print("=" * 78)
    print("\nSIN RAG, el modelo responde de memoria:\n")
    print(sin_rag)
    print("\n  De dónde salió: de los pesos del modelo. No hay nada que puedas verificar.")
    print("\n" + "-" * 78)
    print("\nCON RAG, el modelo lee los fragmentos del libro:\n")
    print(con_rag)
    paginas = ", ".join(str(d.metadata.get("page", "?")) for d in fuentes)
    print(f"\n  De dónde salió: páginas {paginas} del PDF. Puedes ir a comprobarlo.")


comparar("In Alice in Wonderland, who was the White Rabbit hurrying to meet?")

PREGUNTA: In Alice in Wonderland, who was the White Rabbit hurrying to meet?
MODELO:   gemma3:4b en los dos casos

SIN RAG, el modelo responde de memoria:

The White Rabbit was hurrying to meet the **Queen of Hearts**! He was frantically trying to be on time for her croquet game. 😊 

Let me know if you'd like to delve deeper into any other aspect of *Alice in Wonderland*!

  De dónde salió: de los pesos del modelo. No hay nada que puedas verificar.

------------------------------------------------------------------------------

CON RAG, el modelo lee los fragmentos del libro:

According to the text, the White Rabbit was hurrying to meet the Duchess. He kept muttering “Oh ! the Duchess, the Duchess !”

  De dónde salió: páginas 15, 8, 10 del PDF. Puedes ir a comprobarlo.


Lee las dos respuestas con calma, porque aquí está la lección del día.

Las dos suenan igual de seguras. Ninguna duda ni se disculpa. Pero nombran a personajes
distintos, así que por fuerza una está mal.

La forma de salir de dudas no es opinar, es ir al texto. En el capítulo II, cuando el Conejo
Blanco vuelve corriendo con los guantes y el abanico, va murmurando: *"Oh! the Duchess, the
Duchess! Oh! won't she be savage if I've kept her waiting!"*. Está en la página 15 del PDF, y
puedes comprobarlo tú mismo con las celdas de la sección 11. Apunta a la Duquesa.

La respuesta sin recuperación nombra a la Reina de Corazones y agrega que iba a su partida de
croquet. Fíjate de dónde puede salir eso: en la película animada de 1951 el conejo canta que
llega tarde a *una cita muy importante*, y la partida de croquet sí existe en la novela, pero
varios capítulos después y sin relación con esa carrera. El modelo juntó pedazos de la película
y del libro y entregó la mezcla como un hecho, sin avisar.

Ese es el patrón peligroso, y no se parece al que uno imagina. El modelo no dijo un disparate:
dijo algo familiar, plausible y con el tono de quien sabe. Sin haber leído el libro, no tienes
manera de saber a cuál creerle.

El sistema con recuperación acertó por una razón concreta, y es la que da confianza: no estaba
recordando, estaba leyendo. Y como sabes exactamente qué leyó, puedes verificarlo.

Prueba ahora con la consulta del principio. Ojo, la respuesta sin RAG va a ser larga: cuando un
modelo contesta de memoria tiende a extenderse y a llenar los huecos.

In [15]:
comparar("Describe the Mad Hatter's tea party.")

PREGUNTA: Describe the Mad Hatter's tea party.
MODELO:   gemma3:4b en los dos casos

SIN RAG, el modelo responde de memoria:

The Mad Hatter’s tea party, from Lewis Carroll’s *Alice’s Adventures in Wonderland*, is a scene of utter chaos and delightful absurdity – a whirlwind of mismatched decorations, nonsensical conversation, and a general disregard for logic. Here's a breakdown of what makes it so memorable:

**Setting:**

* **A Giant Mushroom Table:** The party takes place around a massive, scarlet mushroom that serves as the table. It’s incredibly large, dwarfing Alice and her companions.
* **Mismatched Furniture:** Scattered around the mushroom are an assortment of bizarre pieces of furniture – a small table, chairs of all shapes and sizes (some upside down!), and even a tiny golden stool. Everything feels deliberately out of place and unsettling. 
* **A Chaotic Decoration Scheme:** The room is covered in a riot of colors: red roses, blue curtains, yellow cushions, and a general f

Compara los detalles concretos, no el estilo. La versión sin recuperación suele describir una
mesa que es un hongo gigante, una habitación llena de cortinas y cojines de colores, y una
celebración del "no cumpleaños".

Nada de eso está en el libro. El texto dice que había *una mesa puesta bajo un árbol, frente a
la casa*. El hongo existe en la novela, pero es el de la Oruga, cuatro capítulos antes. Y la
palabra "unbirthday" no aparece ni una sola vez en el PDF que cargamos: viene otra vez de la
película.

Puedes comprobar cualquiera de esas afirmaciones tú mismo, sin creerme a mí:

```python
texto = " ".join(p.page_content for p in pages)
print(texto.lower().count("unbirthday"))
```

La versión con RAG es más seca y menos vistosa, pero todo lo que afirma se puede rastrear hasta
un fragmento concreto del PDF.

## Y con un modelo más grande, ¿se arregla?

Es la pregunta natural, y vale la pena contestarla con datos en lugar de con intuición. Probé
las dos consultas de arriba con `gemma4:12b`, tres veces más grande que el que acabas de usar.

Mejora, y bastante. No inventa la Reina de Corazones ni el hongo gigante ni el "no cumpleaños".
Sobre la fiesta del té da un resumen correcto, aunque general. Y sobre el Conejo Blanco contesta
que el libro nunca especifica a quién iba a ver.

Fíjate bien en eso último, porque es el punto fino. Esa respuesta prudente también es inexacta:
el libro sí lo dice, en la página 15. El modelo grande dejó de inventar, pero tampoco pudo
darte el dato ni la cita. Cambió una respuesta falsa por una respuesta vaga.

La conclusión es la que va a sostener el resto del curso. Un modelo más grande te compra
prudencia, no conocimiento de tus documentos. Solo la recuperación te da la frase exacta, la
página donde está y la posibilidad de comprobarla.

Un último apunte, para que la conclusión sea justa en el otro sentido. *Alicia en el país de las
maravillas* es de los libros más citados que existen, así que este es el caso más favorable
posible para el modelo sin ayuda: algo sabía. Con el manual interno de una empresa, que no
estuvo en ningún dato de entrenamiento, no habría acertado ni una sola línea. Ese, y no este,
es el escenario en el que vas a trabajar.

## 13. Ahora prueba tú

Cambia la consulta de la celda de abajo y vuelve a ejecutarla. Algunas para empezar:

- `"Who is the Cheshire Cat?"`
- `"What happens when Alice drinks from the bottle?"`
- `"Describe the trial of the Knave of Hearts."`

Después de cada respuesta, vuelve a ejecutar las celdas de la sección 11 para ver qué fragmentos
se usaron. Vas a encontrar preguntas que salen bien y preguntas que salen mal, y casi siempre
la explicación está ahí.

In [16]:
query = "Who is the Cheshire Cat?"
print(rag_chain.invoke(query))

According to the text, the Cheshire Cat is a cat sitting on a bough of a tree who has very long claws and a great many teeth. He only grinned when he saw Alice and was willing to answer her questions about directions. He also appeared and disappeared in parts, with just his grin being visible at times.


## Lo que construiste y lo que falta

Armaste un sistema RAG completo y funciona. Para un primer día, es bastante.

Este es el mismo circuito de la figura del principio, ahora visto como se vería en un sistema
puesto a trabajar de verdad. Las cajas sólidas son las que acabas de construir. Las bandas
punteadas son las piezas que todavía no tiene.

<img src="figuras/02_stack_rag.svg" alt="Diagrama del stack de un sistema RAG en dos carriles: preparar los documentos, que se corre una sola vez, y responder una pregunta, que se corre cada vez. Dos bandas punteadas señalan lo que falta: metadatos, tablas y figuras en el primer carril; búsqueda híbrida, reordenamiento, detección de invenciones y evaluación de la calidad en el segundo." width="900">

Nota algo en la figura: los dos carriles corren en momentos distintos. El de arriba se ejecuta
una sola vez por documento, y es el que tardó al principio de la sesión. El de abajo se ejecuta
en cada pregunta, y es el rápido. El índice es el único punto donde se tocan: el primero lo
escribe, el segundo lo lee.

Sobre lo que falta, para que sepas dónde va cada pieza. La búsqueda híbrida y el reordenamiento
entran dentro de "Buscar", y sirven para que lleguen mejores fragmentos. La detección de
invenciones se coloca entre el modelo y la respuesta, como filtro. Los metadatos se aprovechan
al extraer el texto, y permiten filtrar antes de buscar.

También tiene debilidades que en un sistema de verdad, atendiendo clientes reales, no se pueden
dejar pasar. Las vamos a ir resolviendo a lo largo del curso.

El texto entró sucio, con pies de página y cortes de línea del PDF metidos entre los fragmentos.
El tamaño de los fragmentos lo pusimos a ojo, sin comprobar si es el mejor. Traemos siempre tres
fragmentos, sean útiles o no. No tenemos manera de saber si una respuesta es buena más allá de
leerla y opinar. Y no sabemos qué pasaría con documentos en español y preguntas en español, que
es justo el caso que nos interesa.

Ninguno de esos problemas se arregla poniendo un modelo más grande. Se arreglan con ingeniería,
y de eso trata el resto del curso.